In [128]:
#import core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob

# Visual configurations
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")


Libraries imported successfully!


# 1. CCEE PLD Data (Preço Médio Mensal)

In [129]:
# --- DATA INGESTION & INSPECTION ---

# 1. CCEE PLD Data (Preço Médio Mensal)
df_pld = pd.read_excel("../data/raw/Historico_do_Preco_Medio_Mensal.xls")
display(df_pld.head())


C:\Users\Cesar\AppData\Local\Temp\ipykernel_24984\624769961.py:4: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  df_pld = pd.read_excel("../data/raw/Historico_do_Preco_Medio_Mensal.xls")


,MES,SUDESTE,SUL,NORDESTE,NORTE,Unnamed: 5,"*** COM A DISPONIBILIZAÇÃO DO PORTAL DE DADOS ABERTOS DA CCEE, OS DADOS HISTÓRICOS DO PLD FORAM MIGRADOS PARA ESSA NOVA PLATAFORMA. COM ESTA VERSÃO (1º DE OUTUBRO), ENCERRAM-SE AS ATUALIZAÇÕES POR ESTE CANAL. OS DADOS HISTÓRICOS DO PLD JÁ ESTÃO DISPONÍVEIS NO PORTAL DE DADOS ABERTOS DA CCEE, E PODEM SER ACESSADOS NO SEGUINTE LINK: https://dadosabertos.ccee.org.br/organization/preco_liquidacao_diferenca"
0,2001-01-01,56.92,56.92,33.87,33.87,NaN,*** COM A DISPONIBILIZAÇÃO DO PORTAL DE DADOS ...
1,2001-02-01,160.29,154.17,121.47,121.47,NaN,*** COM A DISPONIBILIZAÇÃO DO PORTAL DE DADOS ...
2,2001-03-01,165.97,151.60,154.21,154.21,NaN,*** COM A DISPONIBILIZAÇÃO DO PORTAL DE DADOS ...
3,2001-04-01,252.18,242.66,247.35,247.35,NaN,*** COM A DISPONIBILIZAÇÃO DO PORTAL DE DADOS ...
4,2001-05-01,459.89,420.22,440.99,440.99,NaN,*** COM A DISPONIBILIZAÇÃO DO PORTAL DE DADOS ...


In [130]:
# === CLEANING & FILTERING CCEE PLD DATA ===

# 1. Select only the necessary columns: MES and SUDESTE
df_pld_clean = df_pld[["MES", "SUDESTE"]].copy()

# 2. rename columns standard English naming convention
df_pld_clean.columns = ["date", "pld_se_co"]

# 3. convet date to ddatetime format and set as index
df_pld_clean["date"] = pd.to_datetime(df_pld_clean["date"])
df_pld_clean = df_pld_clean.dropna(subset = ["date"])
df_pld_clean.set_index("date", inplace=True)

# 4. Sort index chronologically and display first/last rows
df_pld_clean = df_pld_clean.sort_index()
start_date = '2017-01-01'
end_date = '2025-09-01'
df_pld_clean = df_pld_clean.loc[start_date:end_date]
display(df_pld_clean)
print(f"\nTotal months loaded for SE/CO submkarket: {len(df_pld_clean)}")




C:\Users\Cesar\AppData\Local\Temp\ipykernel_24984\1893033911.py:12: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_pld_clean.set_index("date", inplace=True)


,pld_se_co
date,
2017-01-01,121.44
2017-02-01,128.43
2017-03-01,216.24
2017-04-01,371.47
2017-05-01,411.49
...,...
2025-05-01,212.58
2025-06-01,234.71
2025-07-01,210.02



Total months loaded for SE/CO submkarket: 105


# 2. ONS Energy Load (Carga de Energia)

In [131]:
# 2. ONS Energy Load (Carga de Energia)
df_load = pd.read_parquet("../data/raw/CARGA_MENSAL.parquet")

display(df_load)
print("\n Unique columns: ", df_load.columns.tolist())

,id_subsistema,nom_subsistema,din_instante,val_cargaenergiamwmed
0,N,Norte,2000-01-31,2397.64408602
1,NE,Nordeste,2000-01-31,5666.02526882
2,S,Sul,2000-01-31,6726.53514785
3,SE,Sudeste/Centro-Oeste,2000-01-31,24463.18615591
4,N,Norte,2000-02-29,2409.03548851
...,...,...,...,...
1287,SE,Sudeste/Centro-Oeste,2026-10-31,0E-8
1288,N,Norte,2026-11-30,0E-8
1289,NE,Nordeste,2026-11-30,0E-8
1290,S,Sul,2026-11-30,0E-8



 Unique columns:  ['id_subsistema', 'nom_subsistema', 'din_instante', 'val_cargaenergiamwmed']


In [132]:
# === CLEANING & FILTERING ONS ENERGY LOAD DATA ===

# 1. filter only for the "SE" SUBSYSTEM
df_load_clean = df_load[df_load["id_subsistema"] == "SE"].copy()

#display(df_load_clean)

# 2. select only date and mwmed
df_load_clean = df_load_clean[["din_instante", "val_cargaenergiamwmed"]]

# 3. rename columns to standard English naming convention
df_load_clean.columns = ["date", "load_seco_mwmed"]

#display(df_load_clean)
#4. Convert 'load_seco_mwmed' to numeric (forcing errors to NaN if any text string exists)
df_load_clean['load_seco_mwmed'] = pd.to_numeric(df_load_clean['load_seco_mwmed'], errors='coerce')

# 5. Convert to date time format
df_load_clean["date"] = pd.to_datetime(df_load_clean["date"])
df_load_clean.set_index("date", inplace=True)
df_load_clean = df_load_clean.sort_index()

# 6. Remove any rows with zero or null load values (like future dates with 0E-8)
df_load_clean = df_load_clean[df_load_clean["load_seco_mwmed"] > 0]

# 6.5. Align dates to Month-Start (MS) to match other datasets
df_load_clean = df_load_clean.resample("MS").mean()

# 7. Apply date cutoff up to September 2025
start_date = '2017-01-01'
end_date = '2025-09-01'
df_load_clean = df_load_clean.loc[start_date:end_date]

display(df_load_clean)

print(f"\nTotal months loaded for SE/CO Load (up to {cutoff_date}): {len(df_load_clean)}")

C:\Users\Cesar\AppData\Local\Temp\ipykernel_24984\3328671807.py:20: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_load_clean.set_index("date", inplace=True)


,load_seco_mwmed
date,
2017-01-01,39881.955585
2017-02-01,40979.009545
2017-03-01,40491.735269
2017-04-01,37739.542232
2017-05-01,36876.019168
...,...
2025-05-01,44249.910236
2025-06-01,42782.282853
2025-07-01,42211.141737



Total months loaded for SE/CO Load (up to 2025-09-01): 105


# 3. ONS Reservoir Storage (EAR) (Energia Armazenada)

In [133]:
df_ear = pd.read_parquet("../data/raw/EAR/EAR_DIARIO_SUBSISTEMA_2017.parquet")
display(df_ear.head())

,id_subsistema,nom_subsistema,ear_data,ear_max_subsistema,ear_verif_subsistema_mwmes,ear_verif_subsistema_percentual
0,NE,NORDESTE,2017-01-01,51809.0,8494.0,16.39
1,N,NORTE,2017-01-01,15041.0,2866.0,19.05
2,SE,SUDESTE,2017-01-01,203343.0,68544.0,33.71
3,S,SUL,2017-01-01,20100.0,12399.0,61.69
4,NE,NORDESTE,2017-01-02,51809.0,8653.0,16.70


In [134]:
# --- INGESTION & RESAMPLING EAR DATA (ALL YEARS) ---

# 1. Find all parquet files in the EAR directory
ear_files = glob.glob("../data/raw/EAR/EAR_DIARIO_SUBSISTEMA_*.parquet")

# 2. Read and concatenate all files into a single DataFrame
df_ear_list = [pd.read_parquet(file) for file in ear_files]
df_ear_full = pd.concat(df_ear_list, ignore_index=True)

# 3. Filter only for the SE subsystem (Sudeste/Centro-Oeste)
df_ear_se = df_ear_full[df_ear_full["id_subsistema"] == "SE"].copy()

# 4. Convert date to datetime format and set as index
df_ear_se["date"] = pd.to_datetime(df_ear_se["ear_data"]) 
df_ear_se.set_index("date", inplace=True)

# 5. Resample daily data to monthly average (Month-Start)
df_ear_monthly = df_ear_se[["ear_verif_subsistema_mwmes"]].resample("MS").mean()

# 6. Rename columns to standard English naming convention
df_ear_monthly.columns = ["ear_seco_mwmes"]

# 7. Sort index and apply the 2017-2025 date interval
df_ear_monthly = df_ear_monthly.sort_index()
start_date = '2017-01-01'
end_date = '2025-09-01'
df_ear_monthly = df_ear_monthly.loc[start_date:end_date]

# Display results
display(df_ear_monthly.head())
print(f"\nTotal months loaded for SE/CO EAR: {len(df_ear_monthly)}")

C:\Users\Cesar\AppData\Local\Temp\ipykernel_24984\1556057770.py:15: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_ear_se.set_index("date", inplace=True)


,ear_seco_mwmes
date,
2017-01-01,69836.612903
2017-02-01,80383.464286
2017-03-01,83858.096774
2017-04-01,84446.433333
2017-05-01,85763.161290



Total months loaded for SE/CO EAR: 105


# 4. ENA

In [135]:
df_ena = pd.read_parquet("../data/raw/ENA/ENA_DIARIO_SUBSISTEMA_2017.parquet")
display(df_ena.head())

,id_subsistema,nom_subsistema,ena_data,ena_bruta_regiao_mwmed,ena_bruta_regiao_percentualmlt,ena_armazenavel_regiao_mwmed,ena_armazenavel_regiao_percentualmlt
0,NE,NORDESTE,2017-01-01,8620.0,62.41,8597.0,62.25
1,N,NORTE,2017-01-01,3044.0,30.32,3022.0,30.10
2,SE,SUDESTE,2017-01-01,32420.0,50.45,31707.0,49.34
3,S,SUL,2017-01-01,14583.0,192.35,14082.0,185.75
4,NE,NORDESTE,2017-01-02,8596.0,62.24,8572.0,62.07


In [136]:

# --- INGESTION & RESAMPLING ENA DATA (ALL YEARS) ---

# 1. Find all parquet files in the ENA directory
ena_files = glob.glob("../data/raw/ENA/ENA_DIARIO_SUBSISTEMA_*.parquet")

# 2. Read and concatenate all files into a single DataFrame
df_ena_list = [pd.read_parquet(file) for file in ena_files]
df_ena_full = pd.concat(df_ena_list, ignore_index=True)

# 3. Filter only for the SE subsystem (Sudeste/Centro-Oeste)
df_ena_se = df_ena_full[df_ena_full["id_subsistema"] == "SE"].copy()

# 4. Convert date to datetime format and set as index
df_ena_se["date"] = pd.to_datetime(df_ena_se["ena_data"]) 
df_ena_se.set_index("date", inplace=True)

# 5. Resample daily data to monthly average (Month-Start)
df_ena_monthly = df_ena_se[["ena_bruta_regiao_mwmed"]].resample("MS").mean()
# 6. Rename columns to standard English naming convention
df_ena_monthly.columns = ["ena_seco_mwmed"]

# 7. Sort index and apply the 2017 to Sep 2025 date interval
df_ena_monthly = df_ena_monthly.sort_index()
start_date = '2017-01-01'
end_date = '2025-09-01'
df_ena_monthly = df_ena_monthly.loc[start_date:end_date]

# Display results
display(df_ena_monthly.head())
print(f"\nTotal months loaded for SE/CO ENA: {len(df_ena_monthly)}")

C:\Users\Cesar\AppData\Local\Temp\ipykernel_24984\2463481364.py:15: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_ena_se.set_index("date", inplace=True)


,ena_seco_mwmed
date,
2017-01-01,43990.903226
2017-02-01,49413.428571
2017-03-01,45626.032258
2017-04-01,39045.966667
2017-05-01,38930.387097



Total months loaded for SE/CO ENA: 105


# 5. ONS Thermal Generation (Geração Térmica)

In [140]:
df_ter = pd.read_parquet("../data/raw/Thermal Generation/GERACAO_TERMICA_DESPACHO_2017.parquet")

display(df_ter.head())

,din_instante,nom_tipopatamar,id_subsistema,nom_subsistema,nom_usina,cod_usinaplanejamento,ceg,val_proggeracao,val_progordemmerito,val_progordemdemeritoref,...,val_verifgfom,val_verifreposicaoperdas,val_verifexportacao,val_fdexp,val_verifreservapotencia,val_atendsatisfatoriorpo,val_verifgsub,val_verifunitcommitment,val_verifconstrainedoff,tip_restricaoeletrica
0,2017-01-01,LEVE,N,NORTE,Aparecida,201.0,UTE.GN.AM.027250-7.02,0.000,0.000,0.000,...,0.000,0.000,0.000,,0.000,0.000,0.000,,0.000,0
1,2017-01-01,LEVE,N,NORTE,Geramar I,73.0,UTE.PE.MA.029705-4.01,0.000,0.000,0.000,...,0.000,0.000,0.000,,0.000,0.000,0.000,,0.000,0
2,2017-01-01,LEVE,N,NORTE,Geramar II,70.0,UTE.PE.MA.029668-6.01,0.000,0.000,0.000,...,0.000,0.000,0.000,,0.000,0.000,0.000,,0.000,0
3,2017-01-01,LEVE,N,NORTE,Jaraqui,204.0,UTE.GN.AM.029272-9.01,55.000,0.000,0.000,...,0.000,0.000,0.000,,0.000,0.000,0.000,,0.000,0
4,2017-01-01,LEVE,N,NORTE,Maranhão III,212.0,UTE.GN.MA.030800-5.01,519.000,519.000,518.800,...,0.000,0.000,0.000,,0.000,0.000,0.000,,0.000,0


In [ ]:
# --- INGESTION & RESAMPLING THERMAL GENERATION DATA (ALL YEARS) ---

# 1. Find all parquet files in the Thermal Generation directory
# Using wildcard to match both yearly and monthly file formats
ter_files = glob.glob("../data/raw/Thermal Generation/GERACAO_TERMICA_*.parquet")

# 2. Read and concatenate all files into a single DataFrame
df_ter_list = [pd.read_parquet(file) for file in ter_files]
df_ter_full = pd.concat(df_ter_list, ignore_index=True)

# 3. Filter only for the SE subsystem (Sudeste/Centro-Oeste)
df_ter_se = df_ter_full[df_ter_full["id_subsistema"] == "SE"].copy()
# 4. Convert date to datetime format
df_ter_se["date"] = pd.to_datetime(df_ter_se["din_instante"]) 

# 4.5. Force generation values to numeric (converts any text/corrupted data to NaN)
df_ter_se["val_proggeracao"] = pd.to_numeric(df_ter_se["val_proggeracao"], errors="coerce")

# 5. Group by date and sum generation across all individual power plants (usinas)
df_ter_total = df_ter_se.groupby("date")["val_proggeracao"].sum().reset_index()

# 6. Set date as index and resample to monthly average (Month-Start)
df_ter_total.set_index("date", inplace=True)
df_ter_monthly = df_ter_total.resample("MS").mean()

# 7. Rename columns to standard English naming convention
df_ter_monthly.columns = ["thermal_gen_seco_mwmed"]

# 8. Sort index and apply the 2017 to Sep 2025 date interval
df_ter_monthly = df_ter_monthly.sort_index()
start_date = '2017-01-01'
end_date = '2025-09-01'
df_ter_monthly = df_ter_monthly.loc[start_date:end_date]

# Display results
display(df_ter_monthly.head())
print(f"\nTotal months loaded for SE/CO Thermal Generation: {len(df_ter_monthly)}")

C:\Users\Cesar\AppData\Local\Temp\ipykernel_24984\3887358585.py:23: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_ter_total.set_index("date", inplace=True)


,thermal_gen_seco_mwmed
date,
2017-01-01,4221.813172
2017-02-01,4145.487351
2017-03-01,5496.185484
2017-04-01,6608.565278
2017-05-01,6202.169355



Total months loaded for SE/CO Thermal Generation: 84


# Final merge

In [141]:
# --- FINAL MERGE: CONSOLIDATING ALL VARIABLES ---

# 1. Sanitize all indices to ensure perfect matching (removes timezones and time artifacts)
df_pld_clean.index = pd.to_datetime(df_pld_clean.index.strftime('%Y-%m-01'))
df_load_clean.index = pd.to_datetime(df_load_clean.index.strftime('%Y-%m-01'))
df_ear_monthly.index = pd.to_datetime(df_ear_monthly.index.strftime('%Y-%m-01'))
df_ena_monthly.index = pd.to_datetime(df_ena_monthly.index.strftime('%Y-%m-01'))
df_ter_monthly.index = pd.to_datetime(df_ter_monthly.index.strftime('%Y-%m-01'))

# 2. Join all datasets using the date index
df_final = df_pld_clean.join(
    [df_load_clean, df_ear_monthly, df_ena_monthly, df_ter_monthly], 
    how="inner"
)

# Display the final consolidated dataset
display(df_final.head())
display(df_final.tail())
print(f"\nFinal analytical dataset ready! Total consolidated months: {len(df_final)}")

,pld_se_co,load_seco_mwmed,ear_seco_mwmes,ena_seco_mwmed,thermal_gen_seco_mwmed
date,,,,,
2017-01-01,121.44,39881.955585,69836.612903,43990.903226,4221.813172
2017-02-01,128.43,40979.009545,80383.464286,49413.428571,4145.487351
2017-03-01,216.24,40491.735269,83858.096774,45626.032258,5496.185484
2017-04-01,371.47,37739.542232,84446.433333,39045.966667,6608.565278
2017-05-01,411.49,36876.019168,85763.161290,38930.387097,6202.169355


,pld_se_co,load_seco_mwmed,ear_seco_mwmes,ena_seco_mwmed,thermal_gen_seco_mwmed
date,,,,,
2023-08-01,69.04,41439.391192,167276.588710,18510.108581,2797.651882
2023-09-01,80.37,44221.082018,155341.004500,17295.988900,2663.945139
2023-10-01,74.84,44696.148931,144270.122871,22523.348871,1299.866263
2023-11-01,84.40,46364.397690,135421.335533,26426.806633,2284.731944
2023-12-01,74.09,45884.391160,127707.250710,28113.617194,3930.266129



Final analytical dataset ready! Total consolidated months: 84
